# সহজ ভাষায় Notebook Guide

এই notebook-এ theory এবং code পাশাপাশি শেখানো হয়েছে। Technical term English-এ থাকবে, আর explanation Bangla-তে—যাতে code-এর language-এর সাথে পরিচিত থেকেও concept সহজে বোঝা যায়।

## কীভাবে ব্যবহার করবেন?

1. Cell উপর থেকে নিচে sequence অনুযায়ী run করুন।
2. Run করার আগে expected output কী হতে পারে লিখে ভাবুন।
3. Output-এর metric, shape এবং visualization explanation-এর সাথে compare করুন।
4. Error হলে import, file path, data shape এবং dependency একে একে check করুন।
5. Notebook শেষে নিজের ভাষায় লিখুন: এটি কোন problem solve করেছে, কীভাবে করেছে এবং limitation কী।

> **Important:** Notebook-এর সব cell successful run হলেই result correct প্রমাণ হয় না; data leakage, wrong assumption এবং misleading metric-ও validate করতে হবে।

# Exploratory Data Analysis (EDA)
## Industrial Equipment Success Score Predictor

এই notebook model train করার আগে data assumptions validate করে। এখানে আমরা:

- Dataset contract, range এবং quality checks করব
- Feature distribution ও target relationship দেখব
- Correlation-কে causation হিসেবে interpret করার ভুল এড়াব
- Categorical group breakdown compare করব
- ১৩টি raw predictor কীভাবে ২৬টি encoded model input হয় তা verify করব
- Modeling-এর আগে hypothesis ও risk লিখব

In [ ]:
import os
import sys
from pathlib import Path

candidates = [
    Path.cwd().parent,
    Path.cwd(),
    Path.cwd() / "week 7" / "Class 2 Project",
]
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / "src").is_dir() and (p / "config.yaml").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Class 2 Project খুঁজে পাওয়া যায়নি। Notebook project বা repo root থেকে খুলুন।")

# Relative config/artifact paths যেন project-এর ভেতরেই থাকে
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.generator import generate_equipment_data
from src.data.preprocessor import DataPreprocessor
from src.config import get_config

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

config = get_config()
SAMPLE_SIZE = min(config.data.n_samples, 2000)
df = generate_equipment_data(n_samples=SAMPLE_SIZE, seed=config.project.random_seed)
raw_predictors = [c for c in df.columns if c not in {"equipment_id", "success_score"}]

print("Project root:", PROJECT_ROOT)
print(f"Dataset shape: {df.shape}; raw predictors: {len(raw_predictors)}")
assert len(raw_predictors) == 13
df.head()

## 1. Dataset Overview & Summary Statistics

In [ ]:
# Basic info
df.info()
print("\n" + "="*50)

# Numeric summary
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols].describe().round(2)

## 2. Target Variable Distribution (Success Score)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
sns.histplot(df['success_score'], kde=True, bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Success Score Distribution')
axes[0].set_xlabel('Success Score')
axes[0].axvline(df['success_score'].mean(), color='red', linestyle='--', label=f'Mean: {df["success_score"].mean():.1f}')
axes[0].legend()

# Box plot
sns.boxplot(y=df['success_score'], ax=axes[1], color='lightcoral')
axes[1].set_title('Success Score Box Plot')

plt.tight_layout()
plt.show()

print(f"Mean: {df['success_score'].mean():.2f}")
print(f"Std:  {df['success_score'].std():.2f}")
print(f"Min:  {df['success_score'].min():.2f}")
print(f"Max:  {df['success_score'].max():.2f}")

## 3. Feature Distributions

In [ ]:
numeric_features = [
    'operating_temperature', 'vibration_level', 'pressure_reading',
    'power_consumption', 'runtime_hours', 'days_since_maintenance',
    'error_count_24h', 'oil_quality_index', 'load_factor', 'ambient_temperature'
]

fig, axes = plt.subplots(5, 2, figsize=(14, 20))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    sns.histplot(df[col], kde=True, bins=40, ax=axes[i], color='teal')
    axes[i].set_title(f'{col} Distribution')
    axes[i].set_xlabel(col)

plt.tight_layout()
plt.show()

## 4. Correlation Matrix

In [ ]:
# Compute correlation matrix for numeric columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations with target
target_corr = corr['success_score'].drop('success_score').sort_values(key=abs, ascending=False)
print("Top correlations with Success Score:")
print(target_corr)

## 5. Categorical Feature Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

categoricals = ['equipment_type', 'manufacturer', 'facility_location']

for i, col in enumerate(categoricals):
    sns.boxplot(data=df, x=col, y='success_score', ax=axes[i], palette='Set2')
    axes[i].set_title(f'Success Score by {col.replace("_", " ").title()}')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Count plots
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for i, col in enumerate(categoricals):
    sns.countplot(data=df, x=col, ax=axes[i], palette='Set2')
    axes[i].set_title(f'{col.replace("_", " ").title()} Count')
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 6. Feature vs Target Scatter Plots

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(14, 20))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    sns.scatterplot(data=df, x=col, y='success_score', ax=axes[i], alpha=0.3, edgecolor=None)
    axes[i].set_title(f'{col} vs Success Score')
    # Add regression line
    sns.regplot(data=df, x=col, y='success_score', ax=axes[i], scatter=False, color='red')

plt.tight_layout()
plt.show()

## 7. Data Quality Checks

In [ ]:
# Missing values
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found.")

# Duplicates
duplicates = df.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")

# Outliers (IQR method)
print("\nOutlier counts (beyond 1.5*IQR):")
for col in numeric_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < (Q1 - 1.5 * IQR)) | (df[col] > (Q3 + 1.5 * IQR))).sum()
    print(f"  {col}: {outliers} ({100*outliers/len(df):.1f}%)")

## 8. Raw-to-Model Feature Contract

EDA raw columns দেখে; model scaled এবং one-hot encoded array দেখে। এই boundary verify না করলে input-shape bug বা feature mismatch সহজে miss হয়।

In [ ]:
preprocessor = DataPreprocessor()
X_encoded, y_scaled = preprocessor.fit_transform(df)

feature_contract = pd.DataFrame({
    "stage": ["Raw predictors", "Numeric after transform", "One-hot columns", "Final model inputs"],
    "count": [
        len(raw_predictors),
        len(preprocessor.numeric_features),
        len(preprocessor.feature_names_after_transform) - len(preprocessor.numeric_features),
        X_encoded.shape[1],
    ],
})
display(feature_contract)
display(pd.DataFrame({"feature": preprocessor.feature_names_after_transform}).head(12))

print("Encoded shape:", X_encoded.shape)
print("Scaled target mean/std:", round(y_scaled.mean(), 4), round(y_scaled.std(), 4))
assert X_encoded.shape == (len(df), 26)
assert np.isfinite(X_encoded).all() and np.isfinite(y_scaled).all()

### Contract Interpretation

১৩টি raw predictor-এর মধ্যে ১০টি numeric। তিনটি categorical column current catalog অনুযায়ী ১৬টি one-hot column দেয়, তাই final width ২৬। Category catalog বদলালে hard-coded width reliable নয়; training code processed array থেকে width infer করবে। Target scaled হওয়ায় model metric-এর unit-ও track করতে হবে।

## 9. Evidence-Based Summary

In [ ]:
strongest = target_corr.reindex(target_corr.abs().sort_values(ascending=False).index).head(5)
category_spread = {
    col: df.groupby(col)["success_score"].mean().max() - df.groupby(col)["success_score"].mean().min()
    for col in categoricals
}

print("Top absolute numeric correlations with success_score:")
print(strongest.round(3).to_string())
print("\nRange of category-group mean scores:")
for name, spread in category_spread.items():
    print(f"  {name}: {spread:.2f} score points")
print(f"\nMissing cells: {int(df.isna().sum().sum())}")
print(f"Duplicate rows: {int(df.duplicated().sum())}")

### Modeling Readiness Checklist

- [ ] Identifier equipment_id model feature থেকে বাদ থাকবে।
- [ ] Split-এর আগে target leakage তৈরি করে এমন derived column নেই।
- [ ] Preprocessor শুধু training data-তে fit হবে; full data-তে fit করা production workflow নয়।
- [ ] Skewed ও clipped features-এর behavior residual analysis-এ revisit করব।
- [ ] Correlation association দেখায়, causation নয়।
- [ ] Category mean difference sample count ও uncertainty ছাড়া final conclusion নয়।
- [ ] Neural network-এর result mean baseline এবং simpler tabular model-এর সাথে compare করব।

> এই EDA notebook learning convenience-এর জন্য full synthetic sample-এ transformer fit করে শুধু shape দেখায়। Production training pipeline train split-এ fit করে leakage এড়াবে।